# PhoBERT-base-v2 clickbait classifier (text only) - Kaggle run

**Before running**

1. Settings -> Accelerator: **GPU T4** (or P100).
2. Settings -> Internet: **On** (needed to clone the repo, `pip install pyvi` and download `vinai/phobert-base-v2`).

The code and processed data are cloned from GitHub. If you prefer, attach a Kaggle dataset containing `architecture/`, `trainer/`, `utils/` and `data/processed/` instead - it is detected automatically and the clone is skipped.

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_phobert.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_phobert.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/phobert")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "phobert"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Train

Early stopping on validation F1 of the clickbait class; the best checkpoint is reloaded before the test evaluation.

In [ ]:
from trainer.train_phobert import Config, run

cfg = Config(
    data_dir=str(DATA_DIR),
    out_dir=str(OUT_DIR),
    max_length=128,
    use_lead=True,        # False = title only
    batch_size=32,
    epochs=10,
    lr=2e-5,
    weighted_loss=True,
    patience=3,
    num_workers=0,        # text-only batches are cheap; workers only add fork noise
    seed=42,
    limit=0,              # set to e.g. 64 for a quick smoke test
)
result = run(cfg)

## Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(result["history"]).set_index("epoch")
display(history.round(4))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
history[["train_loss", "val_loss"]].plot(ax=ax1, marker="o", title="Loss")
history[["val_accuracy", "val_f1", "val_macro_f1"]].plot(ax=ax2, marker="o", title="Validation metrics")
fig.tight_layout()

## Test results

In [ ]:
pd.Series(result["test_metrics"]).round(4).to_frame("test")

In [ ]:
preds = result["test_predictions"]
pd.crosstab(preds["label"], preds["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

### Per source

The clickbait rate varies a lot by source, so check that the model is not just good on the easy sources.

In [ ]:
from trainer.trainer import classification_metrics

def by_group(col):
    rows = {}
    for name, g in preds.groupby(col):
        m = classification_metrics((g["label"] == "clickbait").to_numpy(), (g["pred"] == "clickbait").to_numpy())
        rows[name] = {"n": len(g), "clickbait_rate": (g["label"] == "clickbait").mean(), **m}
    return pd.DataFrame(rows).T.round(3).sort_values("n", ascending=False)

by_group("source")

In [ ]:
by_group("category")

### Most confident mistakes

In [ ]:
pd.set_option("display.max_colwidth", 140)
wrong = preds[preds["label"] != preds["pred"]].copy()
wrong["confidence"] = (wrong["prob_clickbait"] - 0.5).abs() + 0.5
wrong.sort_values("confidence", ascending=False).head(20)[["source", "title", "label", "pred", "prob_clickbait"]]

## Saved files

`best.pt` (model weights), `tokenizer/`, `config.json`, `history.json`, `test_metrics.json`, `test_predictions.csv`. On Kaggle they are under `/kaggle/working/phobert` and appear in the notebook's Output tab after **Save Version**.

In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")